# SPOTIFY AMÉRICAS - EXPORTAÇÃO PARA POWER BI

Este notebook exporta os dados tratados para CSV, prontos para importação no Power BI.

**Arquivos gerados:**
- 01_tabela_fato.csv - Base principal
- 02_dimensao_pais.csv - Métricas por país
- 03_top_100_musicas.csv - Ranking de músicas
- 04_top_50_artistas.csv - Ranking de artistas
- 05_serie_temporal.csv - Evolução semanal
- 06_perfil_musical.csv - Perfis musicais
- 07_analise_sub_regioes.csv - Análise por sub-região
- 08_streams_por_mes.csv - Sazonalidade mensal
- 09_kpis.csv - KPIs para cards
- metadados.json - Documentação

In [ ]:
import pandas as pd
import os
import json
from datetime import datetime
from google.colab import drive

print("=" * 70)
print("EXPORTAÇÃO PARA POWER BI - SPOTIFY AMÉRICAS")
print("=" * 70)

drive.mount('/content/drive')

In [ ]:
# CARREGAR DADOS
caminhos = [
    '/content/drive/MyDrive/spotify_americas_data/spotify_americas_clean.parquet',
    '/content/drive/MyDrive/spotify_americas_clean.parquet',
    '/content/drive/MyDrive/spotify_americas_data/spotify_americas_clean.csv',
    '/content/drive/MyDrive/spotify_americas_clean.csv'
]

df = None
for caminho in caminhos:
    if os.path.exists(caminho):
        if caminho.endswith('.parquet'):
            df = pd.read_parquet(caminho)
        else:
            df = pd.read_csv(caminho)
        print(f"Carregado: {caminho}")
        break

if df is None:
    raise FileNotFoundError("Arquivo de dados não encontrado! Execute o ETL primeiro.")

print(f"Registros: {len(df):,}")
print(f"Países: {df['country'].nunique()}")

In [ ]:
# PASTA DE SAÍDA
output_dir = '/content/drive/MyDrive/spotify_powerbi/'
os.makedirs(output_dir, exist_ok=True)
print(f"Pasta: {output_dir}")

In [ ]:
# TABELA 1 - FATO
colunas = ['track_name', 'artist_names', 'country', 'rank', 'streams_millions', 'week']
for col in ['sub_region', 'year', 'month', 'danceability', 'energy', 'valence']:
    if col in df.columns:
        colunas.append(col)

df_fato = df[[c for c in colunas if c in df.columns]].copy()
df_fato.to_csv(f'{output_dir}01_tabela_fato.csv', index=False)
print(f"01_tabela_fato.csv - {len(df_fato):,} registros")

In [ ]:
# TABELA 2 - DIMENSÃO PAÍS
df_pais = df.groupby('country').agg({
    'streams_millions': ['sum', 'mean'],
    'danceability': 'mean',
    'energy': 'mean',
    'valence': 'mean',
    'track_name': 'count',
    'artist_names': 'nunique'
}).round(3)

df_pais.columns = ['total_streams', 'media_streams', 'dancabilidade', 'energia', 'valencia', 'total_musicas', 'total_artistas']
df_pais = df_pais.reset_index()

if 'sub_region' in df.columns:
    sub_map = df.groupby('country')['sub_region'].first()
    df_pais['sub_regiao'] = df_pais['country'].map(sub_map)

df_pais.to_csv(f'{output_dir}02_dimensao_pais.csv', index=False)
print(f"02_dimensao_pais.csv - {len(df_pais)} países")

In [ ]:
# TABELA 3 - TOP 100 MÚSICAS
df_top_musicas = (df.groupby(['track_name', 'artist_names'])['streams_millions']
                  .sum()
                  .sort_values(ascending=False)
                  .head(100)
                  .reset_index())
df_top_musicas.columns = ['musica', 'artista', 'streams_milhoes']
df_top_musicas.to_csv(f'{output_dir}03_top_100_musicas.csv', index=False)
print(f"03_top_100_musicas.csv - {len(df_top_musicas)} músicas")

In [ ]:
# TABELA 4 - TOP 50 ARTISTAS
df_top_artistas = (df.groupby('artist_names')['streams_millions']
                   .sum()
                   .sort_values(ascending=False)
                   .head(50)
                   .reset_index())
df_top_artistas.columns = ['artista', 'streams_milhoes']
df_top_artistas.to_csv(f'{output_dir}04_top_50_artistas.csv', index=False)
print(f"04_top_50_artistas.csv - {len(df_top_artistas)} artistas")

In [ ]:
# TABELA 5 - SÉRIE TEMPORAL
df['week'] = pd.to_datetime(df['week'])
df_temporal = df.groupby('week').agg({
    'streams_millions': 'sum',
    'rank': 'mean'
}).reset_index()
df_temporal.columns = ['data', 'streams_milhoes', 'rank_medio']
df_temporal.to_csv(f'{output_dir}05_serie_temporal.csv', index=False)
print(f"05_serie_temporal.csv - {len(df_temporal)} semanas")

In [ ]:
# TABELA 6 - PERFIL MUSICAL
if 'music_profile' in df.columns:
    df_perfil = df.groupby('music_profile').agg({
        'streams_millions': 'sum',
        'track_name': 'count'
    }).round(2).reset_index()
    df_perfil.columns = ['perfil_musical', 'total_streams_m', 'total_musicas']
    df_perfil.to_csv(f'{output_dir}06_perfil_musical.csv', index=False)
    print(f"06_perfil_musical.csv - {len(df_perfil)} perfis")

In [ ]:
# TABELA 7 - ANÁLISE POR SUB-REGIÃO
if 'sub_region' in df.columns:
    df_regiao = df.groupby('sub_region').agg({
        'streams_millions': 'sum',
        'danceability': 'mean',
        'energy': 'mean',
        'country': 'nunique'
    }).round(3).reset_index()
    df_regiao.columns = ['sub_regiao', 'total_streams', 'dancabilidade', 'energia', 'total_paises']
    df_regiao.to_csv(f'{output_dir}07_analise_sub_regioes.csv', index=False)
    print(f"07_analise_sub_regioes.csv - {len(df_regiao)} regiões")

In [ ]:
# TABELA 8 - STREAMS POR MÊS
df_mensal = df.groupby(df['week'].dt.to_period('M')).agg({'streams_millions': 'sum'}).reset_index()
df_mensal.columns = ['mes', 'streams_milhoes']
df_mensal['mes'] = df_mensal['mes'].astype(str)
df_mensal.to_csv(f'{output_dir}08_streams_por_mes.csv', index=False)
print(f"08_streams_por_mes.csv - {len(df_mensal)} meses")

In [ ]:
# TABELA 9 - KPIs
kpis = pd.DataFrame({
    'KPI': [
        'Total Streams (M)',
        'Total Músicas',
        'Total Artistas',
        'Total Países',
        'Dançabilidade Média',
        'Energia Média',
        'Positividade Média'
    ],
    'Valor': [
        f"{df['streams_millions'].sum():.1f}",
        f"{df['track_name'].nunique():,}",
        f"{df['artist_names'].nunique():,}",
        f"{df['country'].nunique()}",
        f"{df['danceability'].mean():.3f}",
        f"{df['energy'].mean():.3f}",
        f"{df['valence'].mean():.3f}"
    ]
})
kpis.to_csv(f'{output_dir}09_kpis.csv', index=False)
print(f"09_kpis.csv - {len(kpis)} KPIs")

In [ ]:
# METADADOS
metadados = {
    'projeto': 'Spotify Américas - Power BI',
    'data_exportacao': datetime.now().isoformat(),
    'total_registros': len(df),
    'total_paises': int(df['country'].nunique()),
    'total_musicas': int(df['track_name'].nunique()),
    'total_artistas': int(df['artist_names'].nunique()),
    'periodo_inicio': df['week'].min().isoformat(),
    'periodo_fim': df['week'].max().isoformat(),
    'total_streams_milhoes': float(df['streams_millions'].sum())
}

with open(f'{output_dir}metadados.json', 'w') as f:
    json.dump(metadados, f, indent=2)
print("metadados.json")

In [ ]:
# RESUMO FINAL
print("\n" + "=" * 70)
print("EXPORTAÇÃO CONCLUÍDA COM SUCESSO!")
print("=" * 70)
print(f"\nArquivos salvos em: {output_dir}\n")

for arquivo in sorted(os.listdir(output_dir)):
    print(f"   {arquivo}")